# Square-QDM numerical evidence for Sec. 7

This notebook collects the numerical evidence used by the square-lattice QDM example in the draft.  It follows the same order as the main text:

1. certify the Type-I cage manifold on the $4\times4$ torus;
2. separate the compact local sector from the one-dimensional collective remainder;
3. test local versus collective deformation stability;
4. continue the compact cage along the fixed-width sequence $L_y=4$, $L_x=4N$;
5. compare the exact cage with the energy-matched microcanonical ensemble using the three local witnesses $A_R$, $Z_R$, and $Y_R$.

The fixed-width limit is quasi-one-dimensional.  The true two-dimensional limit remains outside the scope of this notebook.

## Imports and run controls

The default run includes $4\times4$ and $8\times4$ microcanonical points.  Set `RUN_8X4_MICROCANONICAL=False` for a quick smoke test.  All draft figures are exported as PDF and SVG.

In [ ]:
from dataclasses import replace
from itertools import product
from pathlib import Path
import sys
import time

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.linalg as scipy_linalg
from IPython.display import display

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "qlinks").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the qlinks repository root.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from helpers import save_prx_figure, set_revtex_matplotlib_style

from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.builders import SparseHamiltonianBuilder
from qlinks.caging import (
    CageClassificationConfig,
    CageSearchConfig,
    CageSearcher,
    LocalQDMCageSearchConfig,
    LocalWitnessTemplate,
    Quasi1DSequencePoint,
    RobustQDMLocalCageSearchConfig,
    SquareQDMPeriodicProductUnitCell,
    SquareQDMStripTransferMatrix,
    SquareQDMWitnessPlacement,
    adjacent_gap_ratio_report,
    audit_quasi_1d_sequence,
    beta_zero_matching_subspace,
    cage_compatibility_hierarchy_from_hamiltonians,
    cage_finite_size_scorecard,
    cage_jacobian_conditioning_from_hamiltonian,
    certify_local_witness_on_square_qdm_periodic_sequence,
    certify_square_qdm_periodic_product_sequence,
    classify_cage_state,
    commuting_cyclic_symmetry_sector_basis,
    diagnose_boundary_cancellation_matroid,
    diagnose_eigenpair,
    diagnose_local_channel_spectrum,
    directed_transition_witness_template,
    eigenstate_expectations,
    evaluate_square_qdm_classification_witnesses_on_strips,
    gaussian_spectral_filter,
    local_witnesses_from_classification_report,
    materialize_square_qdm_periodic_product_state,
    operator_coefficient_compatibility,
    partition_cage_hamiltonian,
    product_basis_diagonal_phase_factors,
    project_coefficients_to_beta_zero_match,
    project_operator_to_sector,
    project_state_to_sector,
    regional_cage_quotient,
    robust_qdm_local_cage_search,
    scan_square_qdm_beta_zero_energy_density,
    scan_square_qdm_collective_locality_extension,
    scan_square_qdm_periodic_product_cancellation_scaling,
    scan_windowed_operator_annihilators,
    select_microcanonical_window_by_count,
    select_microcanonical_window_by_width,
    spectral_observable_moments,
    subspace_complement_basis,
    thermal_activity_margin_from_samples,
    thermodynamic_energy_window_plan,
)
from qlinks.models import (
    SquareQDMModel,
    qdm_peierls_couplings_from_link_phases,
    qdm_plaquette_link_gauge_matrix,
)
from qlinks.operators import PlaquettePatternOperator

TOL = 1.0e-10
RANK_TOL = 1.0e-9
RANDOM_SEED = 73291
USE_TEX = False
SAVE_FIGURES = True
SAVE_PDF = True
RUN_8X4_MICROCANONICAL = True

PEIERLS_REFERENCE_PHASE = 0.35
PEIERLS_PATH = np.linspace(0.15, 0.55, 5)
MICROCANONICAL_PREFACTORS = (0.50, 0.75, 1.00)
PRIMARY_WINDOW_PREFACTOR = 0.75
SMOOTH_SIGMA_PREFACTOR = 0.75

DATA_DIR = REPO_ROOT / "experimental" / "data" / "square_qdm_draft_evidence"
FIGURE_DIR = DATA_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

set_revtex_matplotlib_style(base_font_size=9, prefer_tex=USE_TEX)

FIGURE_FORMATS = ("pdf", "svg")

def save_figure(fig, stem, *, aliases=(), close=False):
    save_prx_figure(fig, stem, directory=FIGURE_DIR, formats=FIGURE_FORMATS)
    for alias in aliases:
        if alias != stem:
            save_prx_figure(fig, alias, directory=FIGURE_DIR, formats=FIGURE_FORMATS)
    if close:
        plt.close(fig)


def embedded_state(record, hilbert_size):
    state = np.zeros(hilbert_size, dtype=np.complex128)
    state[np.asarray(record.cage_state.support, dtype=np.int64)] = record.cage_state.local_state
    return state


def square_qdm_basis_translation_permutation(model, basis_configs, *, dx=0, dy=0):
    """Return the basis permutation for a physical lattice translation."""
    link_lookup = {}
    for link in model.lattice.links:
        x, y = model.lattice.sites[int(link.source)].cell
        link_lookup[(int(x), int(y), str(link.kind))] = int(link.id)
    transformed = np.zeros_like(basis_configs)
    lx, ly = int(model.lattice.lx), int(model.lattice.ly)
    for link in model.lattice.links:
        x, y = model.lattice.sites[int(link.source)].cell
        target = link_lookup[((int(x) + dx) % lx, (int(y) + dy) % ly, str(link.kind))]
        transformed[:, target] = basis_configs[:, int(link.id)]
    lookup = {
        tuple(int(value) for value in config): index
        for index, config in enumerate(basis_configs)
    }
    return np.asarray(
        [lookup[tuple(int(value) for value in config)] for config in transformed],
        dtype=np.int64,
    )


def fixed_width_cage_momentum(repeats):
    """Momentum branch followed by the repeated compact product cage."""
    return 0, (2 * int(repeats)) % 4


print({
    "repository": str(REPO_ROOT),
    "data_directory": str(DATA_DIR),
    "run_8x4_microcanonical": RUN_8X4_MICROCANONICAL,
    "reference_peierls_phase": PEIERLS_REFERENCE_PHASE,
})

## 1. $4\times4$ Type-I cage census

We record the shell dimensions, boundary-matrix rank and nullity, support size, and full-Hamiltonian residual for the two Type-I manifolds used in the draft.

In [ ]:
square_model = SquareQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    winding_convention="electric",
    coup_kin=1.0,
    coup_pot=1.0,
)

t0 = time.perf_counter()
square_build = square_model.build(
    basis_solver="dfs",
    builder="sparse",
    backend="scipy",
    sort_basis=True,
)
build_seconds = time.perf_counter() - t0

t0 = time.perf_counter()
square_search = CageSearcher.from_model_build_result(
    square_build,
    config=CageSearchConfig(
        search_type="type1",
        tolerance=TOL,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=64,
        ipr_candidate_count=32,
        ipr_random_seed=1234,
    ),
).run()
search_seconds = time.perf_counter() - t0

records_04 = tuple(square_search[(0, 4)])
record_06 = square_search[(0, 6), 0]
states_04 = np.column_stack(
    [embedded_state(record, square_search.hilbert_size) for record in records_04]
)

{
    "winding_sector": (0, 0),
    "hilbert_dimension": square_search.hilbert_size,
    "counts_by_signature": square_search.counts_by_signature,
    "build_seconds": build_seconds,
    "search_seconds": search_seconds,
}

In [ ]:
scorecard_rows = []
for signature, records in (((0, 4), records_04), ((0, 6), (record_06,))):
    for record_index, record in enumerate(records):
        full_state = embedded_state(record, square_search.hilbert_size)
        scorecard = cage_finite_size_scorecard(
            square_build.hamiltonian,
            record.candidate.vertices,
            full_state,
            kinetic=square_build.kinetic,
            actual_support=record.cage_state.support,
            amplitude_tolerance=TOL,
            rank_tolerance=TOL,
            metadata={
                "signature": str(signature),
                "record": record_index,
                "basis_strategy": "IPR postselection",
                "winding_x": 0,
                "winding_y": 0,
            },
        )
        scorecard_rows.append(scorecard.to_summary_dict())

scorecard_table = pd.DataFrame(scorecard_rows)
scorecard_table.to_csv(DATA_DIR / "qdm_4x4_type1_scorecard.csv", index=False)
display(scorecard_table[[
    "signature", "record", "candidate_shell_size", "boundary_rows",
    "boundary_columns", "boundary_rank", "boundary_nullity",
    "boundary_singular_gap", "actual_support_size",
    "internal_residual", "boundary_residual", "relative_eigenpair_residual",
]])

The $(0,4)$ shell is a $84\times48$ boundary problem with nullity nine.  Eight IPR representatives have support four, while the ninth has support 48.  The $(0,6)$ shell is a $100\times32$ boundary problem with nullity one.

## 2. Compact and collective parts of the $(0,4)$ manifold

The eight support-four cages span the compact local sector.  Removing that span from the complete nine-dimensional $(0,4)$ cage manifold leaves one collective direction.

In [ ]:
regional_supports = tuple(record.cage_state.support for record in records_04[:8])
quotient_report = regional_cage_quotient(
    square_build.kinetic,
    regional_supports,
    states_04,
    tolerance=TOL,
)
quotient_overlaps = np.abs(states_04.conj().T @ quotient_report.quotient_basis) ** 2

full_support = tuple(
    sorted(set().union(*(set(record.cage_state.support) for record in records_04)))
)
full_blocks = partition_cage_hamiltonian(square_build.kinetic, full_support)
full_column = {basis_index: column for column, basis_index in enumerate(full_support)}
regional_columns = tuple(
    tuple(full_column[basis_index] for basis_index in support)
    for support in regional_supports
)
matroid_report = diagnose_boundary_cancellation_matroid(
    full_blocks.boundary,
    regional_columns,
    tolerance=TOL,
)

quotient_table = pd.DataFrame([{
    **quotient_report.to_summary_dict(),
    "collective_record_overlap": float(quotient_overlaps[8, 0]),
    "regional_circuit_count": matroid_report.regional_circuit_count,
    "weighted_relative_dependency_dimension": matroid_report.relative_dependency_dimension,
}])
quotient_table.to_csv(DATA_DIR / "qdm_4x4_compact_collective_quotient.csv", index=False)
display(quotient_table.T)

In [ ]:
fig, ax = plt.subplots(figsize=(3.35, 2.35))
ax.bar(["complete cage\nmanifold", "compact span", "collective\nquotient"], [9, 8, 1])
ax.set_ylabel("Dimension")
ax.set_ylim(0, 10)
ax.set_title(r"$(0,4)$ cage-space decomposition")
save_figure(fig, "qdm_4x4_9_equals_8_plus_1")
plt.show()

## 3. Real-space range of the cancellation

For each plaquette term $K_p$, we evaluate $K_p|\psi\rangle$ and ask how large a real-space window is needed before a nontrivial linear combination annihilates the state.  This directly compares the compact and collective representatives without assuming a particular pairwise cancellation pattern.

In [ ]:
term_builder = SparseHamiltonianBuilder(
    backend="scipy",
    dtype=np.complex128,
    on_missing="raise",
)
kinetic_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator])
    for operator in square_build.kinetic_operators
)
potential_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator])
    for operator in square_build.potential_operators
)
all_local_term_matrices = kinetic_term_matrices + potential_term_matrices
plaquette_centers = tuple(
    tuple(float(value) for value in square_model.lattice.plaquette_anchor_cell(int(pid)))
    for pid in square_model.plaquette_ids()
)

radius_scans = {}
for label, state in (
    ("compact record 0", states_04[:, 0]),
    ("collective record 8", states_04[:, 8]),
):
    radius_scans[label] = scan_windowed_operator_annihilators(
        kinetic_term_matrices,
        state,
        plaquette_centers,
        radii=(0, 1, 2),
        periodic_box=(4, 4),
        metric="chebyshev",
        normalize_actions=True,
        action_tolerance=1.0e-12,
        rank_tolerance=TOL,
    )

radius_rows = [
    {"state": label, **point.to_summary_dict()}
    for label, report in radius_scans.items()
    for point in report.points
]
radius_table = pd.DataFrame(radius_rows)
radius_table.to_csv(DATA_DIR / "qdm_4x4_minimum_annihilator_radius.csv", index=False)
display(radius_table[[
    "state", "radius", "minimum_residual", "n_active", "rank", "nullity",
    "coefficient_support_size", "active_operator_indices",
]])

In [ ]:
fig, ax = plt.subplots(figsize=(3.35, 2.45))
for label, report in radius_scans.items():
    ax.semilogy(
        [point.radius for point in report.points],
        [max(point.minimum_residual, 1.0e-16) for point in report.points],
        marker="o",
        label=label,
    )
ax.axhline(TOL, linestyle="--", linewidth=0.8, label="numerical tolerance")
ax.set_xlabel("Allowed Chebyshev radius")
ax.set_ylabel("Minimum annihilation residual")
ax.set_xticks([0, 1, 2])
ax.legend(frameon=False)
save_figure(fig, "qdm_4x4_annihilator_radius")
plt.show()

The compact representative first reaches the numerical kernel at radius one using two active plaquette terms.  The collective quotient retains a residual about $0.675$ at radius one and reaches a kernel only when all 16 plaquette terms are available.  On the $4\times4$ torus, this is evidence for a system-scale cancellation, not a proof that its radius must diverge on every possible continuation.

## 4. Deformation tests

We compare local changes of plaquette-flip amplitudes, Peierls phases, and plaquette potentials.  The purpose is to identify which perturbation directions preserve the compact and collective cages, and to test a finite Peierls-phase path explicitly.

In [ ]:
term_builder = SparseHamiltonianBuilder(
    backend="scipy",
    dtype=np.complex128,
    on_missing="raise",
)
kinetic_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator]).astype(np.complex128)
    for operator in square_build.kinetic_operators
)
potential_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator]).astype(np.complex128)
    for operator in square_build.potential_operators
)
if len(potential_term_matrices) != 2 * len(square_model.plaquette_ids()):
    raise RuntimeError("Expected two orientation projectors per square plaquette.")
plaquette_potential_matrices = tuple(
    potential_term_matrices[2 * index] + potential_term_matrices[2 * index + 1]
    for index in range(len(square_model.plaquette_ids()))
)
phase_tangent_operators = tuple(
    PlaquettePatternOperator.qdm_flip(
        layout=square_model.layout,
        lattice=square_model.lattice,
        plaquette_id=int(plaquette_id),
        coefficient=1.0j,
        reverse_coefficient=-1.0j,
    )
    for plaquette_id in square_model.plaquette_ids()
)
phase_tangent_matrices = tuple(
    term_builder.build(square_build.basis, [operator]).astype(np.complex128)
    for operator in phase_tangent_operators
)
all_local_term_matrices = kinetic_term_matrices + potential_term_matrices
plaquette_centers = tuple(
    tuple(float(value) for value in square_model.lattice.plaquette_anchor_cell(int(pid)))
    for pid in square_model.plaquette_ids()
)

compact_state = states_04[:, 0]
collective_state = states_04[:, 8]
deformation_alphabets = {
    "flip amplitudes": kinetic_term_matrices,
    "Peierls phases": phase_tangent_matrices,
    "flippability potentials": plaquette_potential_matrices,
    "amplitudes + phases": kinetic_term_matrices + phase_tangent_matrices,
}
state_targets = {
    "compact": compact_state,
    "collective": collective_state,
}

hierarchy_reports = {}
hierarchy_rows = []
singular_rows = []
conditioning_rows = []
for target_name, state in state_targets.items():
    support = np.flatnonzero(np.abs(state) > TOL)
    conditioning = cage_jacobian_conditioning_from_hamiltonian(
        square_build.hamiltonian,
        support,
        state,
        tolerance=TOL,
    )
    conditioning_rows.append({
        "target": target_name,
        **conditioning.to_summary_dict(),
    })
    for alphabet_name, perturbations in deformation_alphabets.items():
        report = cage_compatibility_hierarchy_from_hamiltonians(
            square_build.hamiltonian,
            perturbations,
            support,
            state,
            coefficient_field="real",
            tolerance=TOL,
        )
        hierarchy_reports[(target_name, alphabet_name)] = report
        hierarchy_rows.append({
            "target": target_name,
            "alphabet": alphabet_name,
            "obstruction_rank": report.first_order.rank,
            **report.to_summary_dict(),
        })
        singular_rows.extend(
            {
                "target": target_name,
                "alphabet": alphabet_name,
                "singular_index": singular_index,
                "singular_value": float(singular_value),
            }
            for singular_index, singular_value in enumerate(report.first_order.singular_values)
        )

hierarchy_table = pd.DataFrame(hierarchy_rows)
conditioning_table = pd.DataFrame(conditioning_rows)
singular_table = pd.DataFrame(singular_rows)
hierarchy_table.to_csv(DATA_DIR / "qdm_4x4_cage_obstruction_hierarchy.csv", index=False)
conditioning_table.to_csv(DATA_DIR / "qdm_4x4_cage_conditioning.csv", index=False)
singular_table.to_csv(DATA_DIR / "qdm_4x4_cage_obstruction_spectra.csv", index=False)
display(hierarchy_table)
display(conditioning_table[["target", "support_size", "cage_gap", "full_residual"]])

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 3.2))
plot_table = hierarchy_table.copy()
x = np.arange(len(deformation_alphabets))
width = 0.36
for target_index, target_name in enumerate(("compact", "collective")):
    group = plot_table[plot_table["target"] == target_name].set_index("alphabet").loc[list(deformation_alphabets)]
    ax.bar(
        x + (target_index - 0.5) * width,
        group["first_order_compatible_dimension"],
        width=width,
        label=target_name,
    )
ax.set_xticks(x, list(deformation_alphabets), rotation=16, ha="right")
ax.set_ylabel("First-order compatible dimension")
ax.legend(frameon=False)
save_figure(fig, "qdm_4x4_deformation_compatible_dimensions")
plt.show()

fig, ax = plt.subplots(figsize=(5.8, 3.2))
for (target_name, alphabet_name), group in singular_table.groupby(["target", "alphabet"], sort=False):
    if alphabet_name not in ("flip amplitudes", "Peierls phases"):
        continue
    ax.semilogy(
        group["singular_index"] + 1,
        np.maximum(group["singular_value"], 1.0e-16),
        marker="o",
        label=f"{target_name}: {alphabet_name}",
    )
ax.set_xlabel("Obstruction singular-value index")
ax.set_ylabel("Singular value")
ax.legend(frameon=False, fontsize=7)
save_figure(fig, "qdm_4x4_deformation_obstruction_spectra")
plt.show()

### Uniform Peierls-phase path

The uniform plaquette phase is not generated by a periodic link rephasing on the $4\times4$ torus.  We therefore use it as a physical deformation.  The compact cage remains exact along this path, while the collective representative is lifted.

In [ ]:
gauge_incidence = qdm_plaquette_link_gauge_matrix(square_model.lattice)
gauge_rank = int(np.linalg.matrix_rank(gauge_incidence, tol=RANK_TOL))
uniform_phase_pattern = np.ones(len(square_model.plaquette_ids()), dtype=np.float64)
gauge_projection = gauge_incidence @ np.linalg.lstsq(
    gauge_incidence,
    uniform_phase_pattern,
    rcond=RANK_TOL,
)[0]
uniform_non_gauge_residual = float(np.linalg.norm(uniform_phase_pattern - gauge_projection))

peierls_rows = []
for phase in np.linspace(-0.70, 0.70, 15):
    phase_model = replace(square_model, coup_kin=np.exp(1.0j * phase))
    phase_build = phase_model.build(
        basis_solver="dfs",
        builder="sparse",
        backend="scipy",
        sort_basis=True,
    )
    np.testing.assert_array_equal(phase_build.basis.states, square_build.basis.states)
    for target_name, state in state_targets.items():
        support = np.flatnonzero(np.abs(state) > TOL)
        eigenpair = diagnose_eigenpair(phase_build.hamiltonian, state)
        conditioning = cage_jacobian_conditioning_from_hamiltonian(
            phase_build.hamiltonian,
            support,
            state,
            tolerance=TOL,
        )
        peierls_rows.append({
            "phase": float(phase),
            "target": target_name,
            "energy": float(eigenpair.energy.real),
            "residual": eigenpair.residual_norm,
            "Delta_cage": conditioning.cage_gap,
        })

peierls_path_table = pd.DataFrame(peierls_rows)
peierls_path_table.to_csv(DATA_DIR / "qdm_4x4_uniform_peierls_path.csv", index=False)
pd.DataFrame([{
    "n_plaquette_phases": gauge_incidence.shape[0],
    "n_link_phases": gauge_incidence.shape[1],
    "link_gauge_phase_rank": gauge_rank,
    "uniform_phase_distance_from_link_gauge_subspace": uniform_non_gauge_residual,
}]).to_csv(DATA_DIR / "qdm_4x4_peierls_gauge_rank.csv", index=False)
display(peierls_path_table)

fig, ax = plt.subplots(figsize=(3.5, 2.55))
for target_name, group in peierls_path_table.groupby("target"):
    ax.semilogy(
        group["phase"],
        np.maximum(group["residual"], 1.0e-16),
        marker="o",
        label=target_name,
    )
ax.axhline(TOL, linestyle="--", linewidth=0.8, label="tolerance")
ax.set_xlabel(r"Uniform plaquette phase $\phi$")
ax.set_ylabel("Fixed-vector residual")
ax.legend(frameon=False)
save_figure(fig, "qdm_uniform_peierls_compact_collective_residual")
plt.show()

fig, ax = plt.subplots(figsize=(3.5, 2.55))
for target_name, group in peierls_path_table.groupby("target"):
    ax.plot(group["phase"], group["Delta_cage"], marker="o", label=target_name)
ax.set_xlabel(r"Uniform plaquette phase $\phi$")
ax.set_ylabel(r"Cage-conditioning gap $\Delta_{\rm cage}$")
ax.legend(frameon=False)
save_figure(fig, "qdm_uniform_peierls_cage_gap")
plt.show()

The rank data and the nonlinear phase path answer different questions. Peierls-phase tangents allow the compact state to continue throughout the complete 16-dimensional phase alphabet at first order. The collective state has one first-order phase obstruction, while the remaining directions mostly rotate its amplitudes rather than preserving the original vector. Along the uniform non-gauge phase, the compact state stays exact but the collective quotient acquires a linear residual.

## 5. Exact fixed-width cage sequence

The certified local stripe unit is repeated along $x$, giving a $(4N)\times4$ sequence.  We verify exact cancellation for the undeformed model and at the Peierls reference point $\phi=0.35$ used below.

In [ ]:
local_search_config = LocalQDMCageSearchConfig(
    halo_layers=0,
    boundary_mode="relaxed",
    prune_inactive_local_basis_states=True,
    tolerance=TOL,
    degenerate_basis_strategy="ipr",
    ipr_random_seed=1234,
)
robust_config = RobustQDMLocalCageSearchConfig(
    local_config=local_search_config,
    region_strategies=("stripe",),
    stripe_widths=(1,),
    stripe_directions=(0, 1),
    max_regions_per_strategy=None,
    block_signatures=((0, 2),),
    max_records_per_region=2,
    min_blocks=2,
    max_blocks=None,
    max_product_support_size=2048,
    max_paddings_per_stage=100,
    max_paddings_per_packing=10,
    include_sectors=True,
    padding_stages=("static",),
    tolerance=1.0e-9,
    store_full_states=False,
)

stripe_certified, stripe_context = robust_qdm_local_cage_search(
    square_model,
    config=robust_config,
    return_context=True,
)

repeatable_candidates = []
for report_index, report in enumerate(stripe_certified.reports):
    try:
        candidate = SquareQDMPeriodicProductUnitCell.from_padding(
            square_model,
            stripe_context.blocks,
            report.padding,
            repeat_axis="x",
        )
        certificate = certify_square_qdm_periodic_product_sequence(candidate)
    except ValueError:
        continue
    if certificate.is_certified:
        repeatable_candidates.append((report_index, candidate, certificate))

if not repeatable_candidates:
    raise RuntimeError("No x-repeatable square-QDM product unit cell was found.")

repeatable_report_index, product_unit_cell, product_sequence = repeatable_candidates[0]
{
    "is_certified": product_sequence.is_certified,
    "repeat_axis": product_unit_cell.repeat_axis,
    "energy_density": product_sequence.energy_density,
    "support_size_per_unit_cell": product_unit_cell.support_size_per_unit_cell,
    "unit_cell_winding_sector": product_sequence.unit_cell_winding_sector,
    "verification_repeats": product_sequence.verification_repeats,
}

peierls_product_unit_cell = product_unit_cell.with_couplings(
    coup_kin=np.exp(1.0j * PEIERLS_REFERENCE_PHASE),
    coup_pot=1.0,
)
peierls_product_sequence = certify_square_qdm_periodic_product_sequence(
    peierls_product_unit_cell,
    tolerance=1.0e-9,
)
if not peierls_product_sequence.is_certified:
    raise RuntimeError("The reference Peierls sequence failed exact certification.")
print({
    "peierls_phase": PEIERLS_REFERENCE_PHASE,
    "peierls_sequence_certified": peierls_product_sequence.is_certified,
    "peierls_energy_density": peierls_product_sequence.energy_density,
})


In [ ]:
product_scaling = scan_square_qdm_periodic_product_cancellation_scaling(
    product_unit_cell,
    repeat_counts=(1, 2, 3),
    max_support_size=128,
    tolerance=1.0e-9,
)
product_scaling_table = pd.DataFrame([
    point.to_summary_dict() for point in product_scaling.points
])
product_scaling_table.to_csv(DATA_DIR / "qdm_4N_by_4_exact_sequence.csv", index=False)
display(product_scaling_table[[
    "repeats", "system_size", "support_size", "shell_size",
    "boundary_nullity", "interference_gap", "product_state_boundary_residual",
    "kinetic_constraint_rank", "kinetic_compatible_dimension",
    "kinetic_compatible_fraction", "potential_constraint_rank",
]])

In [ ]:
fig, ax = plt.subplots(figsize=(3.35, 2.45))
ax.plot(product_scaling_table["repeats"], product_scaling_table["interference_gap"], marker="o")
ax.set_xlabel("Number of repeated unit cells $N$")
ax.set_ylabel(r"Boundary singular gap $\Delta_B$")
ax.set_xticks(product_scaling_table["repeats"])
save_figure(fig, "qdm_strip_interference_gap")
plt.show()

fig, ax = plt.subplots(figsize=(3.35, 2.45))
ax.plot(product_scaling_table["repeats"], product_scaling_table["kinetic_constraint_rank"], marker="o", label="compatibility rank")
ax.plot(product_scaling_table["repeats"], [16*n for n in product_scaling_table["repeats"]], marker="o", label="local kinetic parameters")
ax.set_xlabel("Number of repeated unit cells $N$")
ax.set_ylabel("Dimension")
ax.set_xticks(product_scaling_table["repeats"])
ax.legend(frameon=False)
save_figure(fig, "qdm_strip_compatibility_scaling")
plt.show()

The repeated cage remains exact on the tested sizes.  Its local cancellation rule is size independent, although the number of independent local coupling constraints grows with the strip length.  This distinction is kept separate from the ETH test below.

## 6. Three bounded local witnesses: $A_R$, $Z_R$, and $Y_R$

The compact stripe cage provides the same three witness routes used in the spin-1 example.

- $A_R$ is the directed transition from the two local parent configurations into one interference-zero configuration.
- $Z_R=A_R+A_R^\dagger$ is the Hermitian kinetic witness.
- $Y_R=F_{p_0}-F_{p_2}$ is a local potential witness built from two plaquette-flippability projectors in the same stripe cell.  The two plaquettes have equal flippability on every support configuration of the exact cage, so $Y_R|\psi_{\rm cage}\rangle=0$.

All three are normalized so that $\|Q_R\|_\infty=1$, with $Q_R^A=A_R^\dagger A_R$, $Q_R^Z=Z_R^2$, and $Q_R^Y=Y_R^2$.

In [ ]:
stripe_record = stripe_certified.records[repeatable_report_index]
stripe_classification = classify_cage_state(
    stripe_record.cage_state,
    kinetic_matrix=stripe_certified.kinetic_matrix,
    basis_configs=stripe_certified.basis.states,
    hilbert_size=stripe_certified.hilbert_size,
    config=CageClassificationConfig(sector_policy="infer_support_component"),
)
stripe_witnesses = local_witnesses_from_classification_report(stripe_classification)

# Z_R: select the first certified Hermitian local interference witness and
# normalize it to ||Z_R||=1, hence ||Z_R^2||=1.
strip_lengths = (4, 8, 12, 16, 24, 32, 48, 64, 96, 128)
strip_witness_report = evaluate_square_qdm_classification_witnesses_on_strips(
    stripe_classification,
    model=square_model,
    lengths=strip_lengths,
    winding_sector=(0, 0),
    normalization="operator_norm",
    winding_projection="fourier",
)
selected_strip_witness = strip_witness_report.records[0]
z_reference_witness = selected_strip_witness.witness
z_placement = selected_strip_witness.placement


def directed_witness_from_hermitian_star(z_witness):
    """Recover A_R from the local Hermitian star Z_R=A_R+A_R^dagger."""
    operator = np.asarray(z_witness.template.local_operator, dtype=np.complex128)
    adjacency = np.abs(operator) > TOL
    degrees = np.sum(adjacency, axis=1)
    target_index = int(np.argmax(degrees))
    source_indices = np.flatnonzero(adjacency[target_index])
    if source_indices.size < 2:
        raise RuntimeError("Expected a two-parent local interference witness.")
    template = directed_transition_witness_template(
        target_pattern=z_witness.template.local_patterns[target_index],
        source_patterns=[z_witness.template.local_patterns[index] for index in source_indices],
        amplitudes=[operator[target_index, index] for index in source_indices],
        metadata={"name": "A_R", "source": "QDM compact-cage boundary row"},
        normalization="operator_norm",
    )
    return template.instantiate(z_witness.variable_indices)


def plaquette_local_positions(model, global_variable_indices, anchor):
    plaquette_id = next(
        int(pid)
        for pid in model.plaquette_ids()
        if tuple(model.lattice.plaquette_anchor_cell(int(pid))) == tuple(anchor)
    )
    position = {int(variable): index for index, variable in enumerate(global_variable_indices)}
    plaquette_variables = [
        int(model.layout.link_variable_index(int(link_id)))
        for link_id in model.lattice.plaquette_links(plaquette_id)
    ]
    try:
        return tuple(position[variable] for variable in plaquette_variables)
    except KeyError as exc:
        raise RuntimeError(
            f"Plaquette {anchor} is not contained in the selected local witness support."
        ) from exc


def shifted_potential_difference_witness(z_witness):
    """Construct Y_R=F_(0,0)-F_(0,2) on the same local stripe support as Z_R."""
    p0 = plaquette_local_positions(square_model, z_witness.variable_indices, (0, 0))
    p2 = plaquette_local_positions(square_model, z_witness.variable_indices, (0, 2))
    local_patterns = tuple(product((0, 1), repeat=z_witness.template.n_variables))

    def flippable(pattern, positions):
        plaquette_pattern = tuple(pattern[index] for index in positions)
        return 1.0 if plaquette_pattern in ((1, 0, 1, 0), (0, 1, 0, 1)) else 0.0

    diagonal = np.asarray(
        [flippable(pattern, p0) - flippable(pattern, p2) for pattern in local_patterns],
        dtype=np.complex128,
    )
    template = LocalWitnessTemplate(
        pattern_key=(),
        local_patterns=local_patterns,
        local_operator=np.diag(diagonal),
        metadata={
            "name": "Y_R",
            "definition": "F_(0,0)-F_(0,2)",
            "plaquette_anchors": ((0, 0), (0, 2)),
        },
    ).normalized("operator_norm")
    return template.instantiate(z_witness.variable_indices)


a_reference_witness = directed_witness_from_hermitian_star(z_reference_witness)
a_placement = SquareQDMWitnessPlacement.from_local_witness(square_model, a_reference_witness)
y_reference_witness = shifted_potential_difference_witness(z_reference_witness)
y_placement = SquareQDMWitnessPlacement.from_local_witness(square_model, y_reference_witness)

# Verify exact darkness on the repeated cage, both before and after the
# reference Peierls deformation.
sequence_certificates = []
for label, witness in (("A", a_reference_witness), ("Z", z_reference_witness), ("Y", y_reference_witness)):
    base_certificate = certify_local_witness_on_square_qdm_periodic_sequence(
        product_sequence, witness, normalization="operator_norm"
    )
    phase_certificate = certify_local_witness_on_square_qdm_periodic_sequence(
        peierls_product_sequence, witness, normalization="operator_norm"
    )
    sequence_certificates.append({
        "witness": label,
        "base_residual": base_certificate.annihilation_residual,
        "peierls_residual": phase_certificate.annihilation_residual,
        "Q_norm": base_certificate.witness.q_operator_norm,
    })
three_witness_certificate_table = pd.DataFrame(sequence_certificates)
three_witness_certificate_table.to_csv(DATA_DIR / "qdm_three_witness_certificates.csv", index=False)
display(three_witness_certificate_table)


In [ ]:
# Exact beta-zero strip traces are kept as an analytical reference.  The
# energy-matched microcanonical ensemble in the next section is the actual ETH
# comparator for the lambda=1 cage energy.
transfer = SquareQDMStripTransferMatrix(circumference=4)
strip_rows = []
for witness_name, placement in (("A", a_placement), ("Z", z_placement), ("Y", y_placement)):
    scaling = transfer.scan_witness(
        placement,
        lengths=strip_lengths,
        boundary_x="periodic",
        winding_sector=(0, 0),
        winding_projection="fourier",
    )
    for evaluation in scaling.evaluations:
        strip_rows.append({
            "witness": witness_name,
            "length": evaluation.length,
            "circumference": evaluation.circumference,
            "thermal_activity": evaluation.expectation,
            "cage_activity": 0.0,
            "partition_count": evaluation.partition_count,
            "window_width": evaluation.window_width,
        })
three_witness_strip_table = pd.DataFrame(strip_rows)
three_witness_strip_table.to_csv(DATA_DIR / "qdm_three_witness_beta_zero_strip.csv", index=False)
display(three_witness_strip_table.head(12))

fig, ax = plt.subplots(figsize=(3.35, 2.45))
for witness_name, marker in (("A", "o"), ("Z", "s"), ("Y", "^")):
    subset = three_witness_strip_table[three_witness_strip_table["witness"] == witness_name]
    ax.plot(subset["length"], subset["thermal_activity"], marker=marker, label=rf"$Q_R^{witness_name}$")
ax.axhline(0.0, linestyle="--", linewidth=0.8, label="cage")
ax.set_xlabel(r"Strip length $L_x$")
ax.set_ylabel(r"$\mathrm{Tr}(\rho_{\beta=0}Q_R)$")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
save_figure(fig, "qdm_three_witness_beta_zero_activity")
plt.show()


The beta-zero traces show that none of the three positive observables is forced to vanish by the dimer constraint or the winding sector.  They are only a reference check; the ETH comparison at the cage energy is the microcanonical calculation below.

## 7. Energy-matched fixed-width microcanonical ensemble

For each $(4N)\times4$ cage we work in the same zero-winding and momentum sector as the cage and center the energy window at its exact energy.  The window width is $\Delta E=c\sqrt{4L_x}$, so the energy-density width vanishes as $L_x^{-1/2}$.

Because the local witnesses do not generally preserve momentum, the positive observables are projected directly as $P_kQ_RP_k$.  For the Hermitian witnesses we also record the projected first moment and the equilibrium measurement variance.

In [ ]:
def fixed_width_microcanonical_point(repeats, *, phase=PEIERLS_REFERENCE_PHASE):
    raw_instance = product_unit_cell.with_couplings(
        coup_kin=np.exp(1.0j * phase),
        coup_pot=1.0,
    ).instantiate(int(repeats))
    finite_model = replace(raw_instance.model, winding_x=0, winding_y=0)
    instance = replace(raw_instance, model=finite_model)
    t0 = time.perf_counter()
    print(f"  building {finite_model.lx}x{finite_model.ly} zero-winding sector ...", flush=True)
    build = finite_model.build(
        basis_solver="dfs",
        builder="bitmask",
        backend="scipy",
        sort_basis=True,
    )
    configs = basis_configs_from_build_result(build)
    cage = materialize_square_qdm_periodic_product_state(instance, configs)
    cage_report = diagnose_eigenpair(build.hamiltonian, cage)

    tx = square_qdm_basis_translation_permutation(finite_model, configs, dx=1)
    ty = square_qdm_basis_translation_permutation(finite_model, configs, dy=1)
    kx, ky = fixed_width_cage_momentum(repeats)
    sector = commuting_cyclic_symmetry_sector_basis(
        (tx, ty),
        orders=(finite_model.lx, finite_model.ly),
        momentum_indices=(kx, ky),
        labels={"winding_x": 0, "winding_y": 0, "kx_index": kx, "ky_index": ky},
    )
    cage_sector = project_state_to_sector(cage, sector)
    projection_norm = float(np.linalg.norm(cage_sector))
    if projection_norm <= TOL:
        raise RuntimeError("The predicted momentum branch has zero cage weight.")
    cage_sector /= projection_norm

    h_sector = project_operator_to_sector(build.hamiltonian, sector)
    energies, vectors = scipy_linalg.eigh(h_sector, check_finite=False)
    overlaps = np.abs(vectors.conj().T @ cage_sector) ** 2
    scar_index = int(np.argmax(overlaps))
    target_energy = float(cage_report.energy.real)
    degenerate_mask = np.abs(energies - target_energy) <= 1.0e-8

    z_witness = z_placement.instantiate_on_model(finite_model)
    a_witness = a_placement.instantiate_on_model(finite_model)
    y_witness = y_placement.instantiate_on_model(finite_model)

    local_operators = {
        "A": a_witness.embed(configs),
        "Z": z_witness.embed(configs),
        "Y": y_witness.embed(configs),
    }
    q_full = {
        name: operator.conj().T @ operator
        for name, operator in local_operators.items()
    }
    q_sector = {
        name: project_operator_to_sector(operator, sector)
        for name, operator in q_full.items()
    }
    z_sector = project_operator_to_sector(local_operators["Z"], sector)
    y_sector = project_operator_to_sector(local_operators["Y"], sector)

    cage_activities = {
        name: float(np.vdot(cage_sector, operator @ cage_sector).real)
        for name, operator in q_sector.items()
    }
    for name, value in cage_activities.items():
        if abs(value) > 1.0e-9:
            raise RuntimeError(
                f"exact QDM cage is not dark to {name} at {finite_model.lx}x{finite_model.ly}: "
                f"<Q>={value:.3e}"
            )

    q_expectations = {
        name: eigenstate_expectations(operator, vectors)
        for name, operator in q_sector.items()
    }
    z_expectations = eigenstate_expectations(z_sector, vectors)
    y_expectations = eigenstate_expectations(y_sector, vectors)

    rows = []
    primary = None
    for prefactor in MICROCANONICAL_PREFACTORS:
        plan = thermodynamic_energy_window_plan(
            volume=finite_model.lattice.num_plaquettes,
            energy_density=target_energy / finite_model.lattice.num_plaquettes,
            width_prefactor=prefactor,
            local_energy_scale=1.0,
        )
        window = select_microcanonical_window_by_width(
            energies,
            target_energy=target_energy,
            half_width=plan.half_width,
            degeneracy_tolerance=TOL,
        )
        indices = np.asarray(window.indices, dtype=np.int64)
        z_moments = spectral_observable_moments(
            z_sector, vectors, squared_operator=q_sector["Z"], indices=indices
        )
        y_moments = spectral_observable_moments(
            y_sector, vectors, squared_operator=q_sector["Y"], indices=indices
        )
        row = {
            "repeats": int(repeats),
            "Lx": int(finite_model.lx),
            "Ly": int(finite_model.ly),
            "volume": int(finite_model.lattice.num_plaquettes),
            "winding_x": 0,
            "winding_y": 0,
            "kx_index": int(kx),
            "ky_index": int(ky),
            "full_winding_sector_dimension": int(configs.shape[0]),
            "resolved_sector_dimension": int(sector.sector_dimension),
            "Peierls_phase": float(phase),
            "cage_energy": target_energy,
            "cage_energy_density": target_energy / finite_model.lattice.num_plaquettes,
            "cage_residual": cage_report.residual_norm,
            "cage_projection_norm": projection_norm,
            "scar_max_overlap": float(overlaps[scar_index]),
            "scar_degenerate_subspace_weight": float(np.sum(overlaps[degenerate_mask])),
            "scar_degenerate_level_count": int(np.sum(degenerate_mask)),
            "scar_level_energy": float(energies[scar_index]),
            "cage_QA": cage_activities["A"],
            "cage_QZ": cage_activities["Z"],
            "cage_QY": cage_activities["Y"],
            "window_prefactor": float(prefactor),
            "window_requested_half_width": plan.half_width,
            "window_energy_density_half_width": plan.energy_density_half_width,
            "window_actual_half_width": window.half_width,
            "window_state_count": window.n_states,
            "window_center_offset": window.center_offset,
            "thermal_A_activity": float(np.mean(q_expectations["A"][indices])),
            "thermal_Z_mean": z_moments.mean,
            "thermal_Z_activity": z_moments.second_moment,
            "thermal_Z_variance": z_moments.variance,
            "thermal_Y_mean": y_moments.mean,
            "thermal_Y_activity": y_moments.second_moment,
            "thermal_Y_variance": y_moments.variance,
            "runtime_seconds": time.perf_counter() - t0,
        }
        rows.append(row)
        if abs(prefactor - PRIMARY_WINDOW_PREFACTOR) <= TOL:
            primary = row
    if primary is None:
        raise RuntimeError("PRIMARY_WINDOW_PREFACTOR is absent from the sensitivity list.")

    smooth = gaussian_spectral_filter(
        energies,
        target_energy=target_energy,
        sigma=SMOOTH_SIGMA_PREFACTOR * np.sqrt(finite_model.lattice.num_plaquettes),
    )
    weights = np.asarray(smooth.weights)
    primary.update({
        "smooth_A_activity": float(np.dot(weights, q_expectations["A"])),
        "smooth_Z_activity": float(np.dot(weights, q_expectations["Z"])),
        "smooth_Y_activity": float(np.dot(weights, q_expectations["Y"])),
        "smooth_effective_state_count": smooth.effective_state_count,
    })
    try:
        gap = adjacent_gap_ratio_report(
            energies,
            trim_fraction=0.10,
            degeneracy_tolerance=RANK_TOL,
        )
        primary["mean_gap_ratio"] = gap.mean_ratio
        primary["gap_ratio_count"] = len(gap.ratios)
    except ValueError:
        primary["mean_gap_ratio"] = np.nan
        primary["gap_ratio_count"] = 0

    primary_indices = np.asarray(
        select_microcanonical_window_by_width(
            energies,
            target_energy=target_energy,
            half_width=primary["window_requested_half_width"],
            degeneracy_tolerance=TOL,
        ).indices,
        dtype=np.int64,
    )
    scatter = pd.DataFrame({
        "repeats": int(repeats),
        "Lx": int(finite_model.lx),
        "energy": energies,
        "energy_density": energies / finite_model.lattice.num_plaquettes,
        "Q_A": q_expectations["A"],
        "Q_Z": q_expectations["Z"],
        "Q_Y": q_expectations["Y"],
        "Z_mean": z_expectations,
        "Y_mean": y_expectations,
        "is_max_overlap_vector": np.arange(energies.size) == scar_index,
        "is_microcanonical": np.isin(np.arange(energies.size), primary_indices),
    })
    return rows, primary, scatter


repeat_counts = (1, 2) if RUN_8X4_MICROCANONICAL else (1,)
fixed_width_rows = []
fixed_width_primary = []
fixed_width_scatter = []
for repeats in repeat_counts:
    rows, primary, scatter = fixed_width_microcanonical_point(repeats)
    fixed_width_rows.extend(rows)
    fixed_width_primary.append(primary)
    fixed_width_scatter.append(scatter)

fixed_width_window_table = pd.DataFrame(fixed_width_rows)
fixed_width_primary_table = pd.DataFrame(fixed_width_primary)
fixed_width_scatter_table = pd.concat(fixed_width_scatter, ignore_index=True)
fixed_width_window_table.to_csv(DATA_DIR / "qdm_fixed_width_microcanonical_window_sensitivity.csv", index=False)
fixed_width_primary_table.to_csv(DATA_DIR / "qdm_fixed_width_microcanonical_primary.csv", index=False)
fixed_width_scatter_table.to_csv(DATA_DIR / "qdm_fixed_width_eth_scatter.csv", index=False)
display(fixed_width_primary_table)


In [ ]:
# Fixed-width thermal activities and ETH scatter.
fig, ax = plt.subplots(figsize=(3.6, 2.6))
for column, label, marker in (
    ("thermal_A_activity", r"$Q_R^A=A_R^\dagger A_R$", "o"),
    ("thermal_Z_activity", r"$Q_R^Z=Z_R^2$", "s"),
    ("thermal_Y_activity", r"$Q_R^Y=Y_R^2$", "^"),
):
    ax.plot(fixed_width_primary_table["Lx"], fixed_width_primary_table[column], marker=marker, label=label)
ax.axhline(0.0, linestyle="--", linewidth=0.8, label="cage")
ax.set_xlabel(r"Strip length $L_x$")
ax.set_ylabel("Microcanonical activity")
ax.legend(frameon=False, fontsize=7)
ax.grid(alpha=0.3)
save_figure(fig, "qdm_fixed_width_three_witness_activity")
plt.show()

largest_Lx = int(fixed_width_scatter_table["Lx"].max())
scatter_largest = fixed_width_scatter_table[fixed_width_scatter_table["Lx"] == largest_Lx]
primary_largest = fixed_width_primary_table[fixed_width_primary_table["Lx"] == largest_Lx].iloc[0]
volume_largest = float(primary_largest["volume"])
window_center_density = float(primary_largest["cage_energy_density"])
window_half_density = float(primary_largest["window_actual_half_width"]) / volume_largest

fig, ax = plt.subplots(figsize=(3.6, 2.6))
ax.axvspan(
    window_center_density - window_half_density,
    window_center_density + window_half_density,
    alpha=0.10,
    color="0.5",
    label="microcanonical window",
    zorder=0,
)
ax.axvline(window_center_density, linestyle="--", linewidth=0.7, color="0.45")
for column, label, marker in (
    ("Q_A", r"$Q_R^A$", "o"),
    ("Q_Z", r"$Q_R^Z$", "s"),
    ("Q_Y", r"$Q_R^Y$", "^"),
):
    ax.scatter(scatter_largest["energy_density"], scatter_largest[column], s=11, alpha=0.6, marker=marker, label=label)
ax.scatter(
    [window_center_density], [0.0], marker="*", s=80, edgecolors="black",
    linewidths=0.5, label="exact cage", zorder=5,
)
ax.set_xlabel("Energy density")
ax.set_ylabel("Local witness activity")
ax.legend(frameon=False, fontsize=7)
ax.grid(alpha=0.3)
save_figure(fig, "qdm_fixed_width_eth_scatter_largest")
plt.show()


The exact cage is dark to all three witnesses at every tested size.  The energy-matched microcanonical ensemble has positive $A_R^\dagger A_R$, $Z_R^2$, and $Y_R^2$ activity.  The fixed-width sequence is therefore tested with the same three witness routes as the spin-1 example.  Additional $L_x$ values are still needed before claiming a nonzero thermodynamic lower bound.

## 8. Preserving Peierls-phase scan

At fixed $4\times4$ size we vary the uniform Peierls phase around the reference point and recompute the energy-matched microcanonical activities.  The cage and all three local witnesses are continued along the same preserving path.

In [ ]:
def peierls_thermal_point_4x4(phase):
    raw_instance = product_unit_cell.with_couplings(
        coup_kin=np.exp(1.0j * float(phase)),
        coup_pot=1.0,
    ).instantiate(1)
    phase_model = replace(raw_instance.model, winding_x=0, winding_y=0)
    instance = replace(raw_instance, model=phase_model)
    build = phase_model.build(
        basis_solver="dfs", builder="sparse", backend="scipy", sort_basis=True
    )
    configs = basis_configs_from_build_result(build)
    tx = square_qdm_basis_translation_permutation(phase_model, configs, dx=1)
    ty = square_qdm_basis_translation_permutation(phase_model, configs, dy=1)
    kx, ky = fixed_width_cage_momentum(1)
    sector = commuting_cyclic_symmetry_sector_basis(
        (tx, ty), orders=(4, 4), momentum_indices=(kx, ky)
    )
    cage = materialize_square_qdm_periodic_product_state(instance, configs)
    cage_sector = project_state_to_sector(cage, sector)
    cage_sector /= np.linalg.norm(cage_sector)
    h_sector = project_operator_to_sector(build.hamiltonian, sector)
    energies, vectors = scipy_linalg.eigh(h_sector, check_finite=False)

    witnesses = {
        "A": a_placement.instantiate_on_model(phase_model),
        "Z": z_placement.instantiate_on_model(phase_model),
        "Y": y_placement.instantiate_on_model(phase_model),
    }
    local = {name: witness.embed(configs) for name, witness in witnesses.items()}
    q_full = {name: operator.conj().T @ operator for name, operator in local.items()}
    q_sector = {name: project_operator_to_sector(operator, sector) for name, operator in q_full.items()}
    z_sector = project_operator_to_sector(local["Z"], sector)
    y_sector = project_operator_to_sector(local["Y"], sector)

    cage_report = diagnose_eigenpair(build.hamiltonian, cage)
    cage_energy = float(cage_report.energy.real)
    cage_activities = {
        name: float(np.vdot(cage_sector, operator @ cage_sector).real)
        for name, operator in q_sector.items()
    }
    if max(abs(value) for value in cage_activities.values()) > 1.0e-9:
        raise RuntimeError(f"Peierls path lost witness darkness at phi={phase}: {cage_activities}")

    plan = thermodynamic_energy_window_plan(
        volume=16,
        energy_density=cage_energy / 16.0,
        width_prefactor=PRIMARY_WINDOW_PREFACTOR,
        local_energy_scale=1.0,
    )
    window = select_microcanonical_window_by_width(
        energies,
        target_energy=cage_energy,
        half_width=plan.half_width,
        degeneracy_tolerance=TOL,
    )
    indices = np.asarray(window.indices, dtype=np.int64)
    qa = eigenstate_expectations(q_sector["A"], vectors)
    z_moments = spectral_observable_moments(z_sector, vectors, squared_operator=q_sector["Z"], indices=indices)
    y_moments = spectral_observable_moments(y_sector, vectors, squared_operator=q_sector["Y"], indices=indices)
    smooth_filter = gaussian_spectral_filter(
        energies, target_energy=cage_energy, sigma=SMOOTH_SIGMA_PREFACTOR * np.sqrt(16)
    )
    weights = np.asarray(smooth_filter.weights)
    conditioning = cage_jacobian_conditioning_from_hamiltonian(
        build.hamiltonian,
        np.flatnonzero(np.abs(cage) > TOL),
        cage,
        tolerance=TOL,
    )
    return {
        "phase": float(phase),
        "path_parameter": float(phase - PEIERLS_REFERENCE_PHASE),
        "cage_energy": cage_energy,
        "cage_residual": cage_report.residual_norm,
        "Delta_cage": conditioning.cage_gap,
        "window_state_count": window.n_states,
        "window_energy_density_half_width": plan.energy_density_half_width,
        "cage_QA": cage_activities["A"],
        "cage_QZ": cage_activities["Z"],
        "cage_QY": cage_activities["Y"],
        "thermal_A_activity": float(np.mean(qa[indices])),
        "thermal_Z_mean": z_moments.mean,
        "thermal_Z_activity": z_moments.second_moment,
        "thermal_Z_variance": z_moments.variance,
        "thermal_Y_mean": y_moments.mean,
        "thermal_Y_activity": y_moments.second_moment,
        "thermal_Y_variance": y_moments.variance,
        "smooth_A_activity": float(np.dot(weights, eigenstate_expectations(q_sector["A"], vectors))),
        "smooth_Z_activity": float(np.dot(weights, eigenstate_expectations(q_sector["Z"], vectors))),
        "smooth_Y_activity": float(np.dot(weights, eigenstate_expectations(q_sector["Y"], vectors))),
        "smooth_effective_state_count": smooth_filter.effective_state_count,
    }


peierls_thermal_table = pd.DataFrame([peierls_thermal_point_4x4(phase) for phase in PEIERLS_PATH])
peierls_thermal_table.to_csv(DATA_DIR / "qdm_4x4_peierls_three_witness_path.csv", index=False)
display(peierls_thermal_table)

fig, ax = plt.subplots(figsize=(3.6, 2.6))
for column, label, marker in (
    ("thermal_A_activity", r"$Q_R^A$", "o"),
    ("thermal_Z_activity", r"$Q_R^Z$", "s"),
    ("thermal_Y_activity", r"$Q_R^Y$", "^"),
):
    ax.plot(peierls_thermal_table["phase"], peierls_thermal_table[column], marker=marker, label=label)
ax.axvline(PEIERLS_REFERENCE_PHASE, linestyle="--", linewidth=0.8, color="0.4")
ax.set_xlabel(r"Uniform plaquette phase $\phi$")
ax.set_ylabel("Microcanonical activity")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
save_figure(fig, "qdm_peierls_three_witness_activity")
plt.show()

fig, ax = plt.subplots(figsize=(3.6, 2.6))
ax.plot(peierls_thermal_table["phase"], peierls_thermal_table["thermal_Z_variance"], marker="s", label=r"${\rm Var}(Z_R)$")
ax.plot(peierls_thermal_table["phase"], np.abs(peierls_thermal_table["thermal_Z_mean"]), marker="o", label=r"$|\langle Z_R\rangle|$")
ax.plot(peierls_thermal_table["phase"], peierls_thermal_table["thermal_Y_variance"], marker="^", label=r"${\rm Var}(Y_R)$")
ax.plot(peierls_thermal_table["phase"], np.abs(peierls_thermal_table["thermal_Y_mean"]), marker="v", label=r"$|\langle Y_R\rangle|$")
ax.set_xlabel(r"Uniform plaquette phase $\phi$")
ax.set_ylabel("Hermitian mean / variance")
ax.legend(frameon=False, fontsize=7)
ax.grid(alpha=0.3)
save_figure(fig, "qdm_peierls_hermitian_resolution")
plt.show()


## 9. Compact-versus-collective deformation rank

For the same local kinetic perturbation basis we record the total number of parameters, the number of preserving directions, and the nonzero singular spectrum for the compact and collective representatives.  These are the quantities requested by the deformation note in the draft.

In [ ]:
state_resolved_reports = {}
state_resolved_rows = []
state_resolved_singular_rows = []
state_operator_basis = kinetic_term_matrices + phase_tangent_matrices
for label, state in (("compact", compact_state), ("collective", collective_state)):
    report = operator_coefficient_compatibility(
        state_operator_basis,
        state,
        mode="fixed_vectors",
        tolerance=RANK_TOL,
    )
    state_resolved_reports[label] = report
    state_resolved_rows.append(
        {
            "target": label,
            "n_operators": report.n_operators,
            "compatible_dimension": report.compatible_dimension,
            "obstruction_rank": report.rank,
            "singular_gap": report.singular_gap,
        }
    )
    state_resolved_singular_rows.extend(
        {
            "target": label,
            "singular_index": int(i),
            "singular_value": float(value),
        }
        for i, value in enumerate(report.singular_values)
    )

state_resolved_table = pd.DataFrame(state_resolved_rows)
state_resolved_spectrum_table = pd.DataFrame(state_resolved_singular_rows)
state_resolved_table.to_csv(DATA_DIR / "qdm_state_resolved_preserving_dimensions.csv", index=False)
state_resolved_spectrum_table.to_csv(DATA_DIR / "qdm_state_resolved_singular_spectra.csv", index=False)
display(state_resolved_table)

fig, ax = plt.subplots(figsize=(3.35, 2.45))
for label, marker in (("compact", "o"), ("collective", "s")):
    subset = state_resolved_spectrum_table[state_resolved_spectrum_table["target"] == label]
    ax.semilogy(subset["singular_index"], subset["singular_value"], marker=marker, label=label)
ax.set_xlabel("Singular-value index")
ax.set_ylabel("State-resolved obstruction spectrum")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
save_figure(fig, "qdm_state_resolved_singular_spectra")
plt.show()


## 10. Draft-ready fixed-width and deformation figure

Panel (a) shows the fixed-width microcanonical activity of all three witnesses.  Panel (b) shows their persistence along the preserving Peierls path.

In [ ]:
fig = plt.figure(figsize=(7.0, 3.0))
grid = fig.add_gridspec(1, 2, wspace=0.30)
ax0 = fig.add_subplot(grid[0, 0])
ax1 = fig.add_subplot(grid[0, 1])

for column, label, marker in (
    ("thermal_A_activity", r"$Q_R^A$", "o"),
    ("thermal_Z_activity", r"$Q_R^Z$", "s"),
    ("thermal_Y_activity", r"$Q_R^Y$", "^"),
):
    ax0.plot(fixed_width_primary_table["Lx"], fixed_width_primary_table[column], marker=marker, label=label)
ax0.axhline(0.0, linestyle="--", linewidth=0.8, label="cage")
ax0.set_xlabel(r"Strip length $L_x$")
ax0.set_ylabel("Microcanonical activity")
ax0.grid(alpha=0.3)
ax0.legend(frameon=False, fontsize=6)
ax0.text(0.02, 0.98, "(a)", transform=ax0.transAxes, ha="left", va="top")

for column, label, marker in (
    ("thermal_A_activity", r"$Q_R^A$", "o"),
    ("thermal_Z_activity", r"$Q_R^Z$", "s"),
    ("thermal_Y_activity", r"$Q_R^Y$", "^"),
):
    ax1.plot(peierls_thermal_table["phase"], peierls_thermal_table[column], marker=marker, label=label)
ax1.axvline(PEIERLS_REFERENCE_PHASE, linestyle="--", linewidth=0.8, color="0.4")
ax1.set_xlabel(r"Uniform plaquette phase $\phi$")
ax1.set_ylabel("Microcanonical activity")
ax1.grid(alpha=0.3)
ax1.legend(frameon=False, fontsize=6)
ax1.text(0.02, 0.98, "(b)", transform=ax1.transAxes, ha="left", va="top")

fig.tight_layout()
save_figure(fig, "qdm_fixed_width_and_peierls_three_witness")
plt.show()


## Output manifest

In [ ]:
manifest = sorted(
    str(path.relative_to(REPO_ROOT))
    for path in DATA_DIR.rglob("*")
    if path.is_file()
)
for item in manifest:
    print(item)